In [ ]:
# Repository-relative paths for the anonymized reproduction package.
import os
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'analysis_code').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from within the repository tree.')

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
MODULE_DIR = REPO_ROOT / 'analysis_code' / '03_dtw_phenotypes'
EXTERNAL_DATA_ROOT = Path(os.environ.get('HEATPA_DATA_ROOT', REPO_ROOT / 'external_data'))


In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap

from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_samples,
    silhouette_score,
)
from sklearn.preprocessing import RobustScaler

try:
    from dtaidistance import dtw
except Exception as exc:
    raise RuntimeError("The dtaidistance package is required to regenerate Figure 3.") from exc


# ======================================================
# 1. 全局绘图参数
#    关键：svg.fonttype='none' 可让 SVG 文字保留为 <text>
# ======================================================
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.unicode_minus": False,
        "font.size": 8.5,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.linewidth": 0.8,
        "legend.frameon": False,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "figure.dpi": 150,
    }
)


# ======================================================
# 2. 路径设置
#    Notebook / Spyder 中不要使用 __file__
# ======================================================
ROOT = MODULE_DIR / "data"

RAW_EVENT = ROOT / "all_cities_eventlag_resultspost12.csv"

PACKAGE = ROOT.parent
RESULTS_DIR = PACKAGE / "results_csv"
OUTPUT_DIR = PACKAGE / "outputs"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================
# 3. 聚类参数：保持原始代码一致
# ======================================================
DTW_WINDOW = 3
LINKAGE_METHOD = "ward"
K = 4
MIN_CLUSTER_SIZE = 6
K_RANGE = range(3, 7)
LINKAGE_METHODS = ("ward",)

OUTLIER_METHOD = "percentile"
OUTLIER_PARAM = 95.0
OUTLIER_ABS_FLOOR = 8.0
OUTLIER_HANDLING = "assign_nearest"


# ======================================================
# 4. 颜色设置：保持原始代码一致
# ======================================================
PALETTE = {
    1: "#A93432",
    2: "#E3A000",
    3: "#6BA6C9",
    4: "#234F8C",
    5: "#7A7A7A",
}

HEATMAP_ANCHORS = {
    "red": "#A93432",
    "rose": "#C38673",
    "tan": "#D9BA97",
    "cream": "#DAD8BB",
    "pale_blue": "#BBD4D0",
    "light_blue": "#6BA6C9",
    "blue": "#4C8EBA",
    "dark_blue": "#234F8C",
}

DTW_DISTANCE_CMAP = LinearSegmentedColormap.from_list(
    "dtw_red_yellow_blue",
    [
        HEATMAP_ANCHORS["dark_blue"],
        HEATMAP_ANCHORS["blue"],
        HEATMAP_ANCHORS["pale_blue"],
        HEATMAP_ANCHORS["cream"],
        HEATMAP_ANCHORS["tan"],
        HEATMAP_ANCHORS["rose"],
        HEATMAP_ANCHORS["red"],
    ],
    N=256,
)


# ======================================================
# 5. 城市缩写表
# ======================================================
city_abbr_map = {
    'Abilene': 'ABI',
    'Amarillo': 'AMA',
    'Arlington': 'ARL',
    'Atlanta': 'ATL',
    'Aurora': 'AUR',
    'Austin': 'AUS',
    'Bakersfield': 'BFL',
    'Baltimore': 'BAL',
    'Boston': 'BOS',
    'Cape Coral': 'CAP',
    'Chandler': 'CHD',
    'Charleston': 'CHS',
    'Charlotte': 'CLT',
    'Chicago': 'CHI',
    'Cincinnati': 'CVG',
    'Clearwater': 'CLW',
    'Cleveland': 'CLE',
    'Columbia': 'COL',
    'Columbus': 'CMH',
    'Corpus Christi': 'CRP',
    'Dallas': 'DAL',
    'Denver': 'DEN',
    'Detroit': 'DET',
    'Fort Worth': 'FTW',
    'Fresno': 'FAT',
    'Gilbert': 'GLB',
    'Henderson': 'HND',
    'Hollywood': 'HWD',
    'Houston': 'HOU',
    'Indianapolis': 'IND',
    'Jacksonville': 'JAX',
    'Kansas City': 'KCY',
    'Las Vegas': 'LAS',
    'Long Beach': 'LGB',
    'Los Angeles': 'LAX',
    'Louisville': 'SDF',
    'Lubbock': 'LBB',
    'Mesa': 'MES',
    'Miami': 'MIA',
    'Milwaukee': 'MKE',
    'Minneapolis': 'MSP',
    'Miramar': 'MIR',
    'Nashville': 'BNA',
    'New York': 'NYC',
    'Newark': 'EWR',
    'Oakland': 'OAK',
    'Oklahoma City': 'OKC',
    'Orlando': 'ORL',
    'Overland Park': 'OVP',
    'Palm Bay': 'PMB',
    'Philadelphia': 'PHL',
    'Phoenix': 'PHX',
    'Pittsburgh': 'PIT',
    'Portland': 'PDX',
    'Raleigh': 'RDU',
    'Richmond': 'RIC',
    'Riverside': 'RIV',
    'Sacramento': 'SAC',
    'Salt Lake City': 'SLC',
    'San Antonio': 'SAT',
    'San Bernardino': 'SBD',
    'San Diego': 'SAN',
    'San Francisco': 'SFO',
    'San Jose': 'SJC',
    'Santa Ana': 'SNA',
    'Scottsdale': 'SCT',
    'Seattle': 'SEA',
    'St. Louis': 'STL',
    'Saint Louis': 'STL',
    'St. Petersburg': 'SPG',
    'Saint Petersburg': 'SPG',
    'Tallahassee': 'TLH',
    'Tampa': 'TPA',
    'Tucson': 'TUS',
    'Virginia Beach': 'VAB',
    'Visalia': 'VIS',
    'Washington': 'WAS',
    'Washington DC': 'WAS',
}


def city_abbr(city_name: str) -> str:
    return city_abbr_map.get(city_name, city_name)


# ======================================================
# 6. 基础函数
# ======================================================
def save_figure(fig: plt.Figure, stem: str) -> None:
    png = OUTPUT_DIR / f"{stem}.png"
    svg = OUTPUT_DIR / f"{stem}.svg"

    fig.savefig(
        png,
        dpi=450,
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        svg,
        bbox_inches="tight",
        facecolor="white",
        format="svg",
    )

    plt.close(fig)

    # 检查 SVG 文字是否保留为 <text>
    svg_text = svg.read_text(encoding="utf-8", errors="ignore")
    if "<text" in svg_text:
        print(f"SVG text is editable-friendly: {svg}")
    else:
        print(f"Warning: no <text> tag found. Text may have been converted to paths: {svg}")


def lag_columns(prefix: str) -> list[str]:
    return [f"{prefix}_lag{i}" for i in range(1, 13)]


def zscore_rows(arr: np.ndarray) -> np.ndarray:
    out = np.zeros_like(arr, dtype=float)

    for i in range(arr.shape[0]):
        s = np.nanstd(arr[i])
        if s > 1e-6:
            out[i] = (arr[i] - np.nanmean(arr[i])) / s

    return np.nan_to_num(out)


def _zrow(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=float)
    s = v.std()
    return (v - v.mean()) / s if s > 1e-8 else np.zeros_like(v)


# ======================================================
# 7. 读取 PPML 输入：保持原始 Kansas City 去重逻辑
# ======================================================
def load_ppml_input() -> tuple[pd.DataFrame, list[str], np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_csv(RAW_EVENT).dropna(subset=["estimate"]).copy()

    # 保持原始代码逻辑：Kansas City 若有多个 n_obs，仅保留 n_obs 最大的一组
    kc = df[df["city"] == "Kansas City"]
    if not kc.empty and kc["n_obs"].nunique() > 1:
        keep_n = kc["n_obs"].max()
        df = df.drop(kc[kc["n_obs"] != keep_n].index).reset_index(drop=True)

    cities = sorted(df["city"].unique())
    lags = np.array(sorted(df["lag"].unique()), dtype=int)

    beta = (
        df.pivot(index="city", columns="lag", values="estimate")
        .loc[cities, lags]
        .to_numpy(float)
    )

    pval = (
        df.pivot(index="city", columns="lag", values="p.value")
        .loc[cities, lags]
        .to_numpy(float)
    )

    return df, cities, lags, beta, pval


# ======================================================
# 8. Shape features：保持原始代码一致
# ======================================================
FEAT_NAMES = [
    "peak_lag",
    "peak_val",
    "early_auc",
    "late_auc",
    "ratio_early_late",
    "monotone",
    "decay_slope",
    "n_sig",
]


def shape_features(beta_row: np.ndarray, pval_row: np.ndarray, lags: np.ndarray) -> np.ndarray:
    abs_b = np.abs(beta_row)

    peak_i = int(np.argmax(abs_b))
    peak_lag = int(lags[peak_i])
    peak_val = float(beta_row[peak_i])

    mid_lag = int(lags[len(lags) // 2 - 1]) if len(lags) >= 2 else int(lags[0])

    early = float(np.sum(beta_row[lags <= mid_lag]))
    late = float(np.sum(beta_row[lags > mid_lag]))
    ratio_el = early / (abs(late) + 1e-6)

    diffs = np.diff(beta_row)
    monotone = float(np.sum(np.sign(diffs)))

    x = lags.astype(float)
    y = beta_row.astype(float)

    xm = x - x.mean()
    ym = y - y.mean()

    decay_slope = float(np.sum(xm * ym) / (np.sum(xm**2) + 1e-12))

    n_sig = int(np.sum(pval_row < 0.05))

    return np.array(
        [
            peak_lag,
            peak_val,
            early,
            late,
            ratio_el,
            monotone,
            decay_slope,
            n_sig,
        ],
        dtype=float,
    )


def compute_shape_tables(
    cities: list[str],
    beta: np.ndarray,
    pval: np.ndarray,
    lags: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    raw_shape = np.stack(
        [
            shape_features(beta[i], pval[i], lags)
            for i in range(len(cities))
        ]
    )

    raw_shape = np.nan_to_num(raw_shape, nan=0.0, posinf=0.0, neginf=0.0)

    for j in range(raw_shape.shape[1]):
        lo, hi = np.percentile(raw_shape[:, j], [5, 95])
        raw_shape[:, j] = np.clip(raw_shape[:, j], lo, hi)

    shape_std = RobustScaler().fit_transform(raw_shape)

    raw_df = pd.DataFrame(raw_shape, index=cities, columns=FEAT_NAMES)
    std_df = pd.DataFrame(shape_std, index=cities, columns=FEAT_NAMES)

    return raw_df, std_df


# ======================================================
# 9. Outlier 识别：保持原始代码一致
# ======================================================
def flag_outliers(beta: np.ndarray) -> tuple[np.ndarray, np.ndarray, float, str]:
    maxabs = np.abs(beta).max(axis=1)

    if OUTLIER_METHOD == "percentile":
        threshold = float(np.percentile(maxabs, OUTLIER_PARAM))
        rule_desc = f"|b|_max > P{OUTLIER_PARAM:.0f} ({threshold:.2f})"
    else:
        threshold = np.inf
        rule_desc = "no outlier rule"

    if OUTLIER_ABS_FLOOR is not None:
        floor = float(OUTLIER_ABS_FLOOR)
        extreme_mask = (maxabs > threshold) | (maxabs >= floor)
        rule_desc = f"{rule_desc} OR |b|_max >= {floor:.2f}"
    else:
        extreme_mask = maxabs > threshold

    return extreme_mask, maxabs, threshold, rule_desc


# ======================================================
# 10. DTW 距离矩阵：保持原始 RandomState(42)
# ======================================================
def compute_dtw_matrix(features: np.ndarray, window: int = DTW_WINDOW) -> np.ndarray:
    n = features.shape[0]
    dmat = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            dist = dtw.distance(features[i], features[j], window=window)
            dmat[i, j] = dmat[j, i] = dist

    finite = dmat[np.isfinite(dmat)]
    cap = finite.max() * 10 if finite.size > 0 and finite.max() > 0 else 1.0

    dmat = np.where(np.isfinite(dmat), dmat, cap)

    rng = np.random.RandomState(42)
    jitter = rng.uniform(0, 1e-8, dmat.shape)
    jitter = (jitter + jitter.T) / 2
    np.fill_diagonal(jitter, 0)

    return dmat + jitter


# ======================================================
# 11. 聚类质量与 search_best：保持原始代码一致
# ======================================================
def score_partition(
    dmat: np.ndarray,
    feat_for_euclid: np.ndarray,
    labels: np.ndarray,
) -> tuple[float, float, float]:

    if len(set(labels)) < 2:
        return -1.0, -1.0, np.inf

    try:
        sil = float(silhouette_score(dmat, labels, metric="precomputed"))
    except Exception:
        sil = -1.0

    try:
        ch = float(calinski_harabasz_score(feat_for_euclid, labels))
    except Exception:
        ch = -1.0

    try:
        db = float(davies_bouldin_score(feat_for_euclid, labels))
    except Exception:
        db = np.inf

    return sil, ch, db


def search_best(
    dmat: np.ndarray,
    feat_for_euclid: np.ndarray,
    methods: tuple[str, ...] = LINKAGE_METHODS,
    k_range: range = K_RANGE,
    min_size: int = MIN_CLUSTER_SIZE,
) -> tuple[dict, pd.DataFrame, dict[str, np.ndarray]]:

    condensed = squareform(dmat, checks=False)

    trees: dict[str, np.ndarray] = {}
    for method in methods:
        trees[method] = linkage(condensed, method=method)

    rows = []

    for ms in [min_size, 8, 6, 4, 2]:
        rows = []

        for method, z in trees.items():
            for k in k_range:
                labels = fcluster(z, t=k, criterion="maxclust")
                _, counts = np.unique(labels, return_counts=True)

                if len(counts) < 2 or counts.min() < ms:
                    continue

                sil, ch, db = score_partition(dmat, feat_for_euclid, labels)

                rows.append(
                    {
                        "method": method,
                        "k": int(k),
                        "min_size": int(counts.min()),
                        "max_size": int(counts.max()),
                        "silhouette": sil,
                        "calinski_harabasz": ch,
                        "davies_bouldin": db,
                        "labels": labels,
                    }
                )

        if rows:
            break

    if not rows:
        raise RuntimeError("No admissible DTW partition was found.")

    cand = pd.DataFrame(rows)

    def zscore_col(col: str, higher_better: bool = True) -> np.ndarray:
        v = cand[col].to_numpy(float)

        if np.nanstd(v) < 1e-12:
            return np.zeros_like(v)

        zv = (v - np.nanmean(v)) / np.nanstd(v)

        return zv if higher_better else -zv

    cand["composite"] = (
        zscore_col("silhouette")
        + zscore_col("calinski_harabasz")
        + zscore_col("davies_bouldin", higher_better=False)
    )

    cand = cand.sort_values("composite", ascending=False).reset_index(drop=True)

    forced = cand[
        (cand["method"] == LINKAGE_METHOD)
        & (cand["k"] == K)
    ]

    if forced.empty:
        raise RuntimeError(
            f"The requested main solution {LINKAGE_METHOD}, k={K} was not admissible."
        )

    best = forced.sort_values("composite", ascending=False).iloc[0].to_dict()
    best["Z"] = trees[LINKAGE_METHOD]

    return best, cand, trees


# ======================================================
# 12. 主聚类流程：保持原始代码一致
# ======================================================
def run_clustering_from_ppml() -> dict:
    df, cities, lags, beta, pval = load_ppml_input()

    extreme_mask, maxabs, threshold, rule_desc = flag_outliers(beta)

    beta_use = beta.copy()

    primary = zscore_rows(
        np.where(pval < 0.05, beta_use, 0.0)
    )

    shape_raw, shape_std = compute_shape_tables(cities, beta_use, pval, lags)

    shape_raw.to_csv(RESULTS_DIR / "shape_features_raw.csv")
    shape_std.to_csv(RESULTS_DIR / "shape_features_standardised.csv")

    core_idx = np.where(~extreme_mask)[0]
    extreme_idx = np.where(extreme_mask)[0]

    d_core = compute_dtw_matrix(primary[core_idx], window=DTW_WINDOW)

    best, cand, trees = search_best(d_core, primary[core_idx])

    cand.drop(columns=["labels"]).to_csv(
        RESULTS_DIR / "k_search_table.csv",
        index=False,
    )

    core_labels = np.asarray(best["labels"], dtype=int)

    labels = np.zeros(len(cities), dtype=int)
    labels[core_idx] = core_labels

    # 保持原始代码逻辑：outlier 使用 response-shape correlation 重新分配
    outlier_assignments: dict[str, tuple[int, float]] = {}

    if extreme_idx.size > 0 and OUTLIER_HANDLING == "assign_nearest":
        core_z = np.stack(
            [
                _zrow(beta[i])
                for i in core_idx
            ]
        )

        cid_list = sorted(set(core_labels))

        centroids_z = np.stack(
            [
                core_z[core_labels == cid].mean(axis=0)
                for cid in cid_list
            ]
        )

        for oi in extreme_idx:
            z_o = _zrow(beta[oi])

            rs = np.array(
                [
                    float(np.corrcoef(z_o, centroids_z[j])[0, 1])
                    for j in range(len(cid_list))
                ]
            )

            rs = np.nan_to_num(rs, nan=-1.0)

            target = int(cid_list[int(np.argmax(rs))])

            labels[oi] = target
            outlier_assignments[cities[oi]] = (target, float(np.max(rs)))

    city = pd.DataFrame(
        {
            "city": cities,
            "cluster": labels,
        }
    )

    city["is_outlier"] = extreme_mask.astype(int)
    city["beta_max_abs"] = maxabs

    city["outlier_assigned_by"] = [
        (
            f"{OUTLIER_HANDLING} "
            f"(r={outlier_assignments[c][1]:+.2f})"
        )
        if c in outlier_assignments
        else ""
        for c in cities
    ]

    for name in FEAT_NAMES:
        city[name] = shape_raw[name].to_numpy(float)

    for j, lag in enumerate(lags):
        city[f"beta_lag{int(lag)}"] = beta[:, j]
        city[f"pval_lag{int(lag)}"] = pval[:, j]

    city.to_csv(
        RESULTS_DIR / "city_cluster_optimized.csv",
        index=False,
    )

    # 聚类质量
    sil_core = silhouette_samples(
        d_core,
        core_labels,
        metric="precomputed",
    )

    extreme_set = set(extreme_idx.tolist())

    qrows = []

    for cid in sorted(set(labels)):
        idx = np.where(labels == cid)[0]

        beta_c = beta[idx]
        centroid = beta_c.mean(axis=0)

        corrs = [
            np.corrcoef(beta_c[i], centroid)[0, 1]
            for i in range(len(idx))
            if np.std(beta_c[i]) > 1e-8
            and np.std(centroid) > 1e-8
        ]

        core_member_rows = np.where(
            np.isin(core_idx, idx)
        )[0]

        mean_sil = (
            float(np.mean(sil_core[core_member_rows]))
            if core_member_rows.size
            else np.nan
        )

        n_out = sum(
            1
            for i in idx
            if i in extreme_set
        )

        qrows.append(
            {
                "cluster": int(cid),
                "n": int(len(idx)),
                "n_outliers": int(n_out),
                "tag": f"includes {n_out} outlier(s)" if n_out else "",
                "silhouette": (
                    round(mean_sil, 4)
                    if np.isfinite(mean_sil)
                    else np.nan
                ),
                "mean_corr_to_centroid": round(
                    float(np.mean(corrs)) if corrs else 0.0,
                    4,
                ),
                "rmse_to_centroid": round(
                    float(
                        np.sqrt(
                            np.mean(
                                (beta_c - centroid) ** 2
                            )
                        )
                    ),
                    4,
                ),
            }
        )

    quality = pd.DataFrame(qrows)

    quality.to_csv(
        RESULTS_DIR / "cluster_quality_metrics_optimized.csv",
        index=False,
    )

    d_all = compute_dtw_matrix(primary, window=DTW_WINDOW)

    z = best["Z"]

    core_cities = [
        cities[i]
        for i in core_idx
    ]

    return {
        "df": df,
        "cities": cities,
        "lags": lags,
        "beta": beta,
        "pval": pval,
        "primary": primary,
        "shape_raw": shape_raw,
        "shape_std": shape_std,
        "city": city,
        "quality": quality,
        "ksearch": cand.drop(columns=["labels"]).copy(),
        "d_all": d_all,
        "z": z,
        "core_cities": core_cities,
        "rule_desc": rule_desc,
        "threshold": threshold,
        "outlier_assignments": outlier_assignments,
    }


# ======================================================
# 13. 图表辅助函数
# ======================================================
def ordered_city_table(
    city: pd.DataFrame,
    d_all: np.ndarray,
) -> tuple[pd.DataFrame, np.ndarray]:

    order = []

    for c in [1, 2, 3, 4]:
        idx = city.index[
            city["cluster"] == c
        ].tolist()

        idx_sorted = sorted(
            idx,
            key=lambda i: (
                city.loc[i, "is_outlier"],
                city.loc[i, "city"],
            ),
        )

        order.extend(idx_sorted)

    city_ordered = city.loc[order].reset_index(drop=True)

    return city_ordered, d_all[np.ix_(order, order)]


def dendrogram_color_func(
    z: np.ndarray,
    leaf_names: list[str],
    city_to_cluster: dict[str, int],
):

    n = len(leaf_names)

    node_clusters: dict[int, set[int]] = {}

    for i, name in enumerate(leaf_names):
        node_clusters[i] = {
            int(city_to_cluster[name])
        }

    for i, row in enumerate(z):
        left = int(row[0])
        right = int(row[1])

        node_clusters[n + i] = (
            node_clusters[left]
            | node_clusters[right]
        )

    def _color(node_id: int) -> str:
        clusters = node_clusters.get(
            int(node_id),
            set(),
        )

        if len(clusters) == 1:
            return PALETTE[next(iter(clusters))]

        return "#A9A9A9"

    return _color


# ======================================================
# 14. 只生成 fig_nature_composite
#     只改城市显示文本为缩写，不改聚类结果
# ======================================================
def plot_nature_composite(
    city: pd.DataFrame,
    quality: pd.DataFrame,
    d_all: np.ndarray,
    z: np.ndarray,
    core_cities: list[str],
    beta: np.ndarray,
    primary: np.ndarray,
) -> None:

    ordered, d_sorted = ordered_city_table(city, d_all)

    core_mask = city["is_outlier"].to_numpy() == 0

    d_core = compute_dtw_matrix(
        primary[core_mask],
        window=DTW_WINDOW,
    )

    core_labels = city.loc[
        core_mask,
        "cluster",
    ].to_numpy()

    sil = silhouette_samples(
        d_core,
        core_labels,
        metric="precomputed",
    )

    sil_df = pd.DataFrame(
        {
            "city": city.loc[
                core_mask,
                "city",
            ].to_numpy(),
            "cluster": core_labels,
            "silhouette": sil,
        }
    )

    sil_df.to_csv(
        RESULTS_DIR / "silhouette_samples_core.csv",
        index=False,
    )

    fig = plt.figure(figsize=(14.5, 10.0))

    gs = gridspec.GridSpec(
        3,
        4,
        figure=fig,
        height_ratios=[1.40, 0.95, 1.15],
        hspace=0.58,
        wspace=0.40,
    )

    # --------------------------------------------------
    # a. DTW distance matrix
    # --------------------------------------------------
    ax_h = fig.add_subplot(gs[0, :3])

    im = ax_h.imshow(
        d_sorted,
        cmap=DTW_DISTANCE_CMAP,
        aspect="auto",
    )

    acc = 0

    for c in [1, 2, 3, 4]:
        n = int(
            (ordered["cluster"] == c).sum()
        )

        if acc > 0:
            ax_h.axhline(
                acc - 0.5,
                color="#111111",
                lw=0.9,
            )
            ax_h.axvline(
                acc - 0.5,
                color="#111111",
                lw=0.9,
            )

        acc += n

    ordered_city_abbr = ordered["city"].map(city_abbr)

    ax_h.set_xticks(
        np.arange(len(ordered))
    )
    ax_h.set_yticks(
        np.arange(len(ordered))
    )

    ax_h.set_xticklabels(
        ordered_city_abbr,
        rotation=90,
        fontsize=4.8,
    )

    ax_h.set_yticklabels(
        ordered_city_abbr,
        fontsize=4.8,
    )

    for tick, c in zip(
        ax_h.get_xticklabels(),
        ordered["cluster"],
    ):
        tick.set_color(PALETTE[int(c)])

    for tick, c in zip(
        ax_h.get_yticklabels(),
        ordered["cluster"],
    ):
        tick.set_color(PALETTE[int(c)])

    ax_h.set_title(
        "a  DTW distance matrix sorted by ward-k4 cluster",
        loc="left",
        fontweight="bold",
        fontsize=10,
    )

    cb = fig.colorbar(
        im,
        ax=ax_h,
        fraction=0.018,
        pad=0.012,
    )

    cb.set_label(
        "DTW distance",
        fontsize=8,
    )

    # --------------------------------------------------
    # b. silhouette panel
    # --------------------------------------------------
    ax_s = fig.add_subplot(gs[0, 3])

    y0 = 0
    yticks = []
    ylabels = []

    mean_sil = float(
        sil_df["silhouette"].mean()
    )

    for c in [1, 2, 3, 4]:
        vals = np.sort(
            sil_df.loc[
                sil_df["cluster"] == c,
                "silhouette",
            ].to_numpy()
        )

        y = np.arange(
            y0,
            y0 + len(vals),
        )

        ax_s.barh(
            y,
            vals,
            color=PALETTE[c],
            alpha=0.92,
            height=0.85,
        )

        yticks.append(
            y0 + len(vals) / 2
        )
        ylabels.append(f"C{c}")

        y0 += len(vals) + 2

    ax_s.axvline(
        mean_sil,
        color="#333333",
        lw=0.8,
        ls="--",
    )

    ax_s.set_yticks(yticks)
    ax_s.set_yticklabels(ylabels)

    ax_s.set_xlabel("Silhouette")

    ax_s.set_title(
        f"b  Core-city silhouette\nmean={mean_sil:.3f}",
        loc="left",
        fontweight="bold",
        fontsize=10,
    )

    ax_s.grid(
        axis="x",
        color="#E5E5E5",
        lw=0.5,
    )

    # --------------------------------------------------
    # c-f. Raw lag-response curves
    # --------------------------------------------------
    lags = np.arange(1, 13)

    for pos, c in enumerate([1, 2, 3, 4]):
        ax = fig.add_subplot(gs[1, pos])

        mask = city["cluster"].to_numpy() == c

        vals = beta[mask]

        for y, is_out in zip(
            vals,
            city.loc[
                mask,
                "is_outlier",
            ].to_numpy(),
        ):
            ax.plot(
                lags,
                y,
                color=PALETTE[c],
                alpha=0.14 if not is_out else 0.32,
                lw=0.8,
            )

            ax.scatter(
                lags,
                y,
                s=5,
                color=PALETTE[c],
                alpha=0.18 if not is_out else 0.45,
                lw=0,
            )

        mean = np.nanmean(vals, axis=0)

        ax.plot(
            lags,
            mean,
            color=PALETTE[c],
            lw=2.2,
        )

        ax.axhline(
            0,
            color="#777777",
            lw=0.7,
            ls=":",
        )

        qrow = quality.loc[
            quality["cluster"] == c
        ].iloc[0]

        tag = (
            ""
            if pd.isna(qrow.get("tag", ""))
            else str(qrow.get("tag", ""))
        )

        ax.set_title(
            f"{chr(99 + pos)}  C{c} (n={int(qrow['n'])})  "
            f"sil={float(qrow['silhouette']):.2f}  "
            f"r={float(qrow['mean_corr_to_centroid']):.2f}"
            + (f"\n{tag}" if tag else ""),
            loc="left",
            color=PALETTE[c],
            fontweight="bold",
            fontsize=8.2,
        )

        ax.set_xlim(1, 12)
        ax.set_xticks([1, 3, 5, 7, 9, 11])
        ax.set_xlabel("Lag (days)")

        if pos == 0:
            ax.set_ylabel("PPML coefficient")

        ax.grid(
            axis="y",
            color="#E8E8E8",
            lw=0.55,
        )

    # --------------------------------------------------
    # g. Dendrogram
    # --------------------------------------------------
    ax_d = fig.add_subplot(gs[2, :])

    by_name = city.set_index("city")

    core_city_labels = [
        city_abbr(c)
        for c in core_cities
    ]

    abbr_to_city = {
        city_abbr(c): c
        for c in core_cities
    }

    dendrogram(
        z,
        labels=core_city_labels,
        leaf_rotation=90,
        leaf_font_size=5.8,
        above_threshold_color="#A9A9A9",
        link_color_func=dendrogram_color_func(
            z,
            core_cities,
            by_name["cluster"].astype(int).to_dict(),
        ),
        ax=ax_d,
    )

    for tick in ax_d.get_xticklabels():
        abbr = tick.get_text()
        full_city = abbr_to_city.get(abbr, abbr)

        c = int(
            by_name.loc[
                full_city,
                "cluster",
            ]
        )

        tick.set_color(PALETTE[c])
        tick.set_fontweight("bold")

    for line in ax_d.get_lines():
        line.set_linewidth(0.9)

    outlier_assign = (
        city.loc[
            city["is_outlier"] == 1,
            ["city", "cluster"],
        ]
        .sort_values("city")
    )

    outlier_txt = "; ".join(
        [
            f"{city_abbr(r.city)}->C{int(r.cluster)}"
            for r in outlier_assign.itertuples()
        ]
    )

    ax_d.set_title(
        f"g  Hierarchy on core cities; "
        f"outliers reassigned by nearest response shape: {outlier_txt}",
        loc="left",
        fontweight="bold",
        fontsize=9,
    )

    ax_d.set_ylabel("Ward DTW distance")

    save_figure(
        fig,
        "fig_nature_composite",
    )


# ======================================================
# 15. 主程序：仅生成 fig_nature_composite
# ======================================================
def main() -> None:
    analysis = run_clustering_from_ppml()

    city = analysis["city"]
    quality = analysis["quality"]
    beta = analysis["beta"]
    primary = analysis["primary"]
    d_all = analysis["d_all"]
    z = analysis["z"]
    core_cities = analysis["core_cities"]

    plot_nature_composite(
        city=city,
        quality=quality,
        d_all=d_all,
        z=z,
        core_cities=core_cities,
        beta=beta,
        primary=primary,
    )

    print(f"Saved: {OUTPUT_DIR / 'fig_nature_composite.svg'}")
    print(f"Saved: {OUTPUT_DIR / 'fig_nature_composite.png'}")


if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.transforms import Bbox
from matplotlib.lines import Line2D
from pathlib import Path
from shapely.geometry import LineString


# =========================================================
# 0. 全局参数
# =========================================================

# ---------- 圆形城市点数据 ----------
city_cluster_path = Path(
    str(EXTERNAL_DATA_ROOT / "gis" / "city_cluster.shp")
)

# ---------- 城市名称字段 ----------
CITY_NAME_FIELD = "city_name"

# ---------- 城市缩写映射 ----------
city_abbr_map = {
    'Abilene': 'ABI',
    'Amarillo': 'AMA',
    'Arlington': 'ARL',
    'Atlanta': 'ATL',
    'Aurora': 'AUR',
    'Austin': 'AUS',
    'Bakersfield': 'BFL',
    'Baltimore': 'BAL',
    'Boston': 'BOS',
    'Cape Coral': 'CAP',
    'Chandler': 'CHD',
    'Charleston': 'CHS',
    'Charlotte': 'CLT',
    'Chicago': 'CHI',
    'Cincinnati': 'CVG',
    'Clearwater': 'CLW',
    'Cleveland': 'CLE',
    'Columbia': 'COL',
    'Columbus': 'CMH',
    'Corpus Christi': 'CRP',
    'Dallas': 'DAL',
    'Denver': 'DEN',
    'Detroit': 'DET',
    'Fort Worth': 'FTW',
    'Fresno': 'FAT',
    'Gilbert': 'GLB',
    'Henderson': 'HND',
    'Hollywood': 'HWD',
    'Houston': 'HOU',
    'Indianapolis': 'IND',
    'Jacksonville': 'JAX',
    'Kansas City': 'KCY',
    'Las Vegas': 'LAS',
    'Long Beach': 'LGB',
    'Los Angeles': 'LAX',
    'Louisville': 'SDF',
    'Lubbock': 'LBB',
    'Mesa': 'MES',
    'Miami': 'MIA',
    'Milwaukee': 'MKE',
    'Minneapolis': 'MSP',
    'Miramar': 'MIR',
    'Nashville': 'BNA',
    'New York': 'NYC',
    'Newark': 'EWR',
    'Oakland': 'OAK',
    'Oklahoma City': 'OKC',
    'Orlando': 'ORL',
    'Overland Park': 'OVP',
    'Palm Bay': 'PMB',
    'Philadelphia': 'PHL',
    'Phoenix': 'PHX',
    'Pittsburgh': 'PIT',
    'Portland': 'PDX',
    'Raleigh': 'RDU',
    'Richmond': 'RIC',
    'Riverside': 'RIV',
    'Sacramento': 'SAC',
    'Salt Lake City': 'SLC',
    'San Antonio': 'SAT',
    'San Bernardino': 'SBD',
    'San Diego': 'SAN',
    'San Francisco': 'SFO',
    'San Jose': 'SJC',
    'Santa Ana': 'SNA',
    'Scottsdale': 'SCT',
    'Seattle': 'SEA',
    'St. Louis': 'STL',
    'Saint Louis': 'STL',
    'St. Petersburg': 'SPG',
    'Saint Petersburg': 'SPG',
    'Tallahassee': 'TLH',
    'Tampa': 'TPA',
    'Tucson': 'TUS',
    'Virginia Beach': 'VAB',
    'Visalia': 'VIS',
    'Washington': 'WAS',
    'Washington DC': 'WAS',
}


def format_city_label(city_name):
    """
    将城市名格式化为：
    Full city name (ABBR)

    若缩写表中没有该城市，则保留原城市名。
    """
    if city_name is None:
        return ""

    name = str(city_name).strip()

    if name == "":
        return ""

    abbr = city_abbr_map.get(name)

    if abbr is None:
        return name

    return f"{name} ({abbr})"


# ---------- 棱形覆盖点数据 ----------
overlay_gdb_path = str(EXTERNAL_DATA_ROOT / "gis" / "MyProject1.gdb")
overlay_layer = "city_cluster_ExportFeatures"

# ---------- 美国边界数据 ----------
gdb_path = str(EXTERNAL_DATA_ROOT / "gis" / "MyProject1.gdb")
outline_layer = "Outline"

states_shp = str(EXTERNAL_DATA_ROOT / "gis" / "US_states.shp")

# ---------- 输出文件夹 ----------
out_dir = Path(
    str(MODULE_DIR / "output")
)
out_dir.mkdir(parents=True, exist_ok=True)

output_name = "city_cluster_spatial_layout_full_citynames_with_abbr"

# ---------- 目标投影 ----------
TARGET_CRS = "EPSG:5070"

# ---------- 图形尺寸 ----------
FIG_WIDTH = 10.5
FIG_HEIGHT_RATIO = 0.68
FIG_HEIGHT = FIG_WIDTH * FIG_HEIGHT_RATIO
DPI = 600

# ---------- 字体 ----------
FONT_FAMILY = "Times New Roman"
plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


# =========================================================
# 1. 点图层样式
# =========================================================

# ---------- cluster 颜色 ----------
COLOR_C1 = "#B23A3A"      # red
COLOR_C2 = "#E69F00"      # orange
COLOR_C3 = "#67A9CF"      # light blue
COLOR_C4 = "#1F4E8C"      # dark blue
COLOR_NO_HW = "#9E9E9E"   # gray

# ---------- 美国底图颜色 ----------
USA_FACE_COLOR = "#F0F0F0"
USA_EDGE_COLOR = "#B8B8B8"

# ---------- 空间点大小控制 ----------
NO_HEATWAVE_POINT_SIZE = 120
CLUSTER_POINT_SIZE = 150
OUTLIER_POINT_SIZE = 230

BASE_POINT_STYLES = {
    0: {
        "marker": "s",
        "facecolor": COLOR_NO_HW,
        "edgecolor": "white",
        "size": NO_HEATWAVE_POINT_SIZE,
        "linewidth": 0.8,
        "label": "No composite heatwave (n=12)"
    },
    1: {
        "marker": "o",
        "facecolor": COLOR_C1,
        "edgecolor": "white",
        "size": CLUSTER_POINT_SIZE,
        "linewidth": 0.8,
        "label": "C1 (n=18)"
    },
    2: {
        "marker": "o",
        "facecolor": COLOR_C2,
        "edgecolor": "white",
        "size": CLUSTER_POINT_SIZE,
        "linewidth": 0.8,
        "label": "C2 (n=14)"
    },
    3: {
        "marker": "o",
        "facecolor": COLOR_C3,
        "edgecolor": "white",
        "size": CLUSTER_POINT_SIZE,
        "linewidth": 0.8,
        "label": "C3 (n=11)"
    },
    4: {
        "marker": "o",
        "facecolor": COLOR_C4,
        "edgecolor": "white",
        "size": CLUSTER_POINT_SIZE,
        "linewidth": 0.8,
        "label": "C4 (n=20)"
    },
}

# ---------- 棱形覆盖点，作为 outlier 样本，位于最上层 ----------
OVERLAY_POINT_STYLES = {
    0: {
        "marker": "D",
        "facecolor": COLOR_NO_HW,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
    1: {
        "marker": "D",
        "facecolor": COLOR_C1,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
    2: {
        "marker": "D",
        "facecolor": COLOR_C2,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
    3: {
        "marker": "D",
        "facecolor": COLOR_C3,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
    4: {
        "marker": "D",
        "facecolor": COLOR_C4,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
}


# =========================================================
# 2. 城市标注参数
# =========================================================

SHOW_CITY_LABELS = True

# None 表示标注所有城市
# 若图面太密，可以改为 [1, 2, 3, 4]，不标注 cluster=0
LABEL_CLUSTER_VALUES = None

# ---------- 城市名称字体 ----------
# 全名 + 缩写比原标签更长，因此适当缩小字号
CITY_LABEL_SIZE = 7.2
CITY_LABEL_COLOR = "#8A8A8A"

CITY_LABEL_BBOX = False
CITY_LABEL_BBOX_ALPHA = 0.65

SKIP_OVERLAPPED_LABELS = True
KEEP_LABELS_INSIDE_AXES = True

# ---------- 点位障碍半径，避免文字压住圆点或菱形点 ----------
BASE_POINT_OBSTACLE_RADIUS_PX = 22
OVERLAY_POINT_OBSTACLE_RADIUS_PX = 30

# ---------- 文字框膨胀系数，减少城市名之间重叠 ----------
LABEL_BBOX_EXPAND_X = 1.25
LABEL_BBOX_EXPAND_Y = 1.45

# ---------- 城市名与点位的距离控制 ----------
# 数值越大，城市名离散点越远
LABEL_DISTANCE_SCALE = 1.65

LABEL_CANDIDATE_OFFSETS_BASE = [
    (26000, 18000, "left", "center"),
    (26000, -18000, "left", "center"),
    (-26000, 18000, "right", "center"),
    (-26000, -18000, "right", "center"),

    (38000, 0, "left", "center"),
    (-38000, 0, "right", "center"),
    (0, 36000, "center", "bottom"),
    (0, -36000, "center", "top"),

    (52000, 30000, "left", "center"),
    (52000, -30000, "left", "center"),
    (-52000, 30000, "right", "center"),
    (-52000, -30000, "right", "center"),

    (70000, 0, "left", "center"),
    (-70000, 0, "right", "center"),
    (0, 62000, "center", "bottom"),
    (0, -62000, "center", "top"),

    (90000, 45000, "left", "center"),
    (90000, -45000, "left", "center"),
    (-90000, 45000, "right", "center"),
    (-90000, -45000, "right", "center"),

    # 更远候选位置，适合东部密集城市群
    (115000, 60000, "left", "center"),
    (115000, -60000, "left", "center"),
    (-115000, 60000, "right", "center"),
    (-115000, -60000, "right", "center"),

    (140000, 0, "left", "center"),
    (-140000, 0, "right", "center"),
    (0, 100000, "center", "bottom"),
    (0, -100000, "center", "top"),
]

LABEL_CANDIDATE_OFFSETS = [
    (
        dx * LABEL_DISTANCE_SCALE,
        dy * LABEL_DISTANCE_SCALE,
        ha,
        va
    )
    for dx, dy, ha, va in LABEL_CANDIDATE_OFFSETS_BASE
]


# =========================================================
# 3. 边界、经纬网、图框参数
# =========================================================

OUTLINE_COLOR = "#9A9A9A"
OUTLINE_WIDTH = 0.85

STATE_LINE_COLOR = "#B8B8B8"
STATE_LINE_WIDTH = 0.55

GRID_LONS = [-120, -110, -100, -90, -80]
GRID_LATS = [20, 30, 40, 50]
LAT_LABELS = [30, 40, 50]

GRID_EXTEND_LON_MIN = -132
GRID_EXTEND_LON_MAX = -58
GRID_EXTEND_LAT_MIN = 12
GRID_EXTEND_LAT_MAX = 60

GRID_COLOR = "#D0D0D0"
GRID_WIDTH = 0.45
GRID_ALPHA = 0.90
GRID_STYLE = "-"

ADD_BOTTOM_LABEL_AXIS_LINE = True
BOTTOM_AXIS_LINE_WIDTH = 0.8
BOTTOM_AXIS_LINE_COLOR = "black"
BOTTOM_TICK_LENGTH_RATIO = 0.018
BOTTOM_LABEL_OFFSET_RATIO = 0.025

ADD_LEFT_LABEL_AXIS_LINE = True
LEFT_AXIS_LINE_WIDTH = 0.8
LEFT_AXIS_LINE_COLOR = "black"
LEFT_TICK_LENGTH_RATIO = 0.018
LEFT_LABEL_OFFSET_RATIO = 0.030

SPINE_WIDTH = 0.8

TICK_SIZE = 10
SCALE_TEXT_SIZE = 9

# =========================================================
# 比例尺参数：10 km，并放在分类 legend 上方
# =========================================================
SHOW_SCALEBAR = False
SCALE_LENGTH_M = 10_000
SCALE_HEIGHT_M = 18_000
SCALE_X_FRAC = 0.070
SCALE_Y_FRAC = 0.245
SCALE_TEXT_OFFSET_M = 8_000
SCALE_LABEL_TEXT = "10 km"

# ---------- 基础显示范围边距 ----------
PAD_X_RATIO = 0.015
PAD_Y_RATIO = 0.020

# ---------- 数据框高宽比控制 ----------
DATA_FRAME_ASPECT_RATIO = 0.70
FIX_WIDTH_ADJUST_HEIGHT = True

# =========================================================
# 美国地图在图中的比例控制
# =========================================================
# 数值越大，美国地图在图中越小
# 推荐范围：1.08–1.30
MAP_SHRINK_FACTOR = 1.18

# 可选：单独控制横向和纵向缩放
MAP_SHRINK_FACTOR_X = 1.00
MAP_SHRINK_FACTOR_Y = 1.00

# 可选：地图中心位置微调，单位为显示范围比例
# 正值：向右 / 向上移动；负值：向左 / 向下移动
MAP_CENTER_SHIFT_X_RATIO = 0.00
MAP_CENTER_SHIFT_Y_RATIO = 0.00


# ---------- 图例 ----------
SHOW_LEGEND = False
LEGEND_FONT_SIZE = 10
LEGEND_MARKER_SIZE = 8.5

# 分类 legend 放在左下角，比例尺位于它上方
LEGEND_LOC = "lower left"
LEGEND_BBOX = (0.02, 0.02)

LEGEND_FRAME = True
LEGEND_FRAME_ALPHA = 0.92
LEGEND_EDGE_COLOR = "#D0D0D0"

SAVE_PNG = True
SAVE_SVG = True


# =========================================================
# 4. 辅助函数
# =========================================================

def make_lonlat_line(lon=None, lat=None, n=1200):
    if lon is not None:
        lats = np.linspace(GRID_EXTEND_LAT_MIN, GRID_EXTEND_LAT_MAX, n)
        coords = [(lon, y) for y in lats]
    elif lat is not None:
        lons = np.linspace(GRID_EXTEND_LON_MIN, GRID_EXTEND_LON_MAX, n)
        coords = [(x, lat) for x in lons]
    else:
        raise ValueError("lon 和 lat 至少需要指定一个")

    return gpd.GeoDataFrame(
        geometry=[LineString(coords)],
        crs="EPSG:4326"
    )


def _interp_x_at_y(x, y, y0, xlim):
    xs = []

    for i in range(len(x) - 1):
        y1, y2 = y[i], y[i + 1]
        x1, x2 = x[i], x[i + 1]

        if (y1 - y0) * (y2 - y0) <= 0 and y1 != y2:
            t = (y0 - y1) / (y2 - y1)
            xi = x1 + t * (x2 - x1)

            if xlim[0] <= xi <= xlim[1]:
                xs.append(xi)

    if len(xs) == 0:
        return None

    x_center = (xlim[0] + xlim[1]) / 2
    return min(xs, key=lambda v: abs(v - x_center))


def _interp_y_at_x(x, y, x0, ylim):
    ys = []

    for i in range(len(x) - 1):
        x1, x2 = x[i], x[i + 1]
        y1, y2 = y[i], y[i + 1]

        if (x1 - x0) * (x2 - x0) <= 0 and x1 != x2:
            t = (x0 - x1) / (x2 - x1)
            yi = y1 + t * (y2 - y1)

            if ylim[0] <= yi <= ylim[1]:
                ys.append(yi)

    if len(ys) == 0:
        return None

    y_center = (ylim[0] + ylim[1]) / 2
    return min(ys, key=lambda v: abs(v - y_center))


def add_graticules(ax, target_crs):
    ax.set_xticks([])
    ax.set_yticks([])

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x_range = xlim[1] - xlim[0]
    y_range = ylim[1] - ylim[0]

    lon_lines = {}
    lat_lines = {}

    for lon in GRID_LONS:
        line = make_lonlat_line(lon=lon).to_crs(target_crs)
        x, y = line.geometry.iloc[0].xy
        x = np.asarray(x)
        y = np.asarray(y)
        lon_lines[lon] = (x, y)

        ax.plot(
            x, y,
            color=GRID_COLOR,
            linewidth=GRID_WIDTH,
            alpha=GRID_ALPHA,
            linestyle=GRID_STYLE,
            zorder=1,
            clip_on=True
        )

    for lat in GRID_LATS:
        line = make_lonlat_line(lat=lat).to_crs(target_crs)
        x, y = line.geometry.iloc[0].xy
        x = np.asarray(x)
        y = np.asarray(y)
        lat_lines[lat] = (x, y)

        ax.plot(
            x, y,
            color=GRID_COLOR,
            linewidth=GRID_WIDTH,
            alpha=GRID_ALPHA,
            linestyle=GRID_STYLE,
            zorder=1,
            clip_on=True
        )

    y_axis = ylim[0]
    bottom_tick_len = y_range * BOTTOM_TICK_LENGTH_RATIO
    bottom_label_offset = y_range * BOTTOM_LABEL_OFFSET_RATIO

    if ADD_BOTTOM_LABEL_AXIS_LINE:
        ax.plot(
            [xlim[0], xlim[1]],
            [y_axis, y_axis],
            color=BOTTOM_AXIS_LINE_COLOR,
            linewidth=BOTTOM_AXIS_LINE_WIDTH,
            zorder=20,
            clip_on=False
        )

    for lon in GRID_LONS:
        x_line, y_line = lon_lines[lon]
        x_label = _interp_x_at_y(x_line, y_line, y_axis, xlim)

        if x_label is None:
            continue

        if ADD_BOTTOM_LABEL_AXIS_LINE:
            ax.plot(
                [x_label, x_label],
                [y_axis, y_axis - bottom_tick_len],
                color=BOTTOM_AXIS_LINE_COLOR,
                linewidth=BOTTOM_AXIS_LINE_WIDTH,
                zorder=20,
                clip_on=False
            )

        ax.text(
            x_label,
            y_axis - bottom_label_offset,
            f"{abs(lon):.0f}°W",
            ha="center",
            va="top",
            fontsize=TICK_SIZE,
            clip_on=False,
            zorder=20
        )

    x_axis = xlim[0]
    left_tick_len = x_range * LEFT_TICK_LENGTH_RATIO
    left_label_offset = x_range * LEFT_LABEL_OFFSET_RATIO

    if ADD_LEFT_LABEL_AXIS_LINE:
        ax.plot(
            [x_axis, x_axis],
            [ylim[0], ylim[1]],
            color=LEFT_AXIS_LINE_COLOR,
            linewidth=LEFT_AXIS_LINE_WIDTH,
            zorder=20,
            clip_on=False
        )

    for lat in LAT_LABELS:
        if lat not in lat_lines:
            continue

        x_line, y_line = lat_lines[lat]
        y_label = _interp_y_at_x(x_line, y_line, x_axis, ylim)

        if y_label is None:
            continue

        if ADD_LEFT_LABEL_AXIS_LINE:
            ax.plot(
                [x_axis, x_axis - left_tick_len],
                [y_label, y_label],
                color=LEFT_AXIS_LINE_COLOR,
                linewidth=LEFT_AXIS_LINE_WIDTH,
                zorder=20,
                clip_on=False
            )

        ax.text(
            x_axis - left_label_offset,
            y_label,
            f"{lat:.0f}°N",
            ha="right",
            va="center",
            fontsize=TICK_SIZE,
            clip_on=False,
            zorder=20
        )


def add_scale_bar(ax):
    """
    添加 10 km 比例尺，并将其放在分类 legend 上方。
    位置由 SCALE_X_FRAC 和 SCALE_Y_FRAC 控制。
    """
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x_range = xlim[1] - xlim[0]
    y_range = ylim[1] - ylim[0]

    x0 = xlim[0] + SCALE_X_FRAC * x_range
    y0 = ylim[0] + SCALE_Y_FRAC * y_range

    rect = Rectangle(
        (x0, y0),
        SCALE_LENGTH_M,
        SCALE_HEIGHT_M,
        facecolor="black",
        edgecolor="black",
        linewidth=0,
        zorder=45
    )
    ax.add_patch(rect)

    ax.text(
        x0 + SCALE_LENGTH_M / 2,
        y0 - SCALE_TEXT_OFFSET_M,
        SCALE_LABEL_TEXT,
        ha="center",
        va="top",
        fontsize=SCALE_TEXT_SIZE,
        color="black",
        zorder=45
    )


def get_display_xlim_ylim(outline_5070):
    """
    根据美国边界计算显示范围。

    MAP_SHRINK_FACTOR > 1 时，会扩大 xlim/ylim，
    从视觉上缩小美国地图在画布中的比例。
    """
    xmin, ymin, xmax, ymax = outline_5070.total_bounds

    x_center = (xmin + xmax) / 2
    y_center = (ymin + ymax) / 2

    raw_width = xmax - xmin
    raw_height = ymax - ymin

    width_with_pad = raw_width * (1 + 2 * PAD_X_RATIO)
    height_with_pad = raw_height * (1 + 2 * PAD_Y_RATIO)

    if FIX_WIDTH_ADJUST_HEIGHT:
        target_width = width_with_pad
        target_height = target_width * DATA_FRAME_ASPECT_RATIO
        target_height = max(target_height, height_with_pad)
    else:
        target_height = height_with_pad
        target_width = target_height / DATA_FRAME_ASPECT_RATIO
        target_width = max(target_width, width_with_pad)

    target_width = target_width * MAP_SHRINK_FACTOR * MAP_SHRINK_FACTOR_X
    target_height = target_height * MAP_SHRINK_FACTOR * MAP_SHRINK_FACTOR_Y

    x_center = x_center + target_width * MAP_CENTER_SHIFT_X_RATIO
    y_center = y_center + target_height * MAP_CENTER_SHIFT_Y_RATIO

    xlim = (
        x_center - target_width / 2,
        x_center + target_width / 2
    )

    ylim = (
        y_center - target_height / 2,
        y_center + target_height / 2
    )

    return xlim, ylim


def make_point_obstacles(ax, gdf, radius_px):
    obstacles = []

    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue

        px, py = ax.transData.transform((geom.x, geom.y))

        obstacles.append(
            Bbox.from_extents(
                px - radius_px,
                py - radius_px,
                px + radius_px,
                py + radius_px
            )
        )

    return obstacles


def bbox_inside_axes(bbox, ax_bbox):
    return (
        bbox.x0 >= ax_bbox.x0 and
        bbox.x1 <= ax_bbox.x1 and
        bbox.y0 >= ax_bbox.y0 and
        bbox.y1 <= ax_bbox.y1
    )


def add_city_labels_greedy(ax, city_gdf, name_field, base_point_gdf, overlay_point_gdf):
    if name_field not in city_gdf.columns:
        raise ValueError(f"城市点数据中未找到字段：{name_field}")

    if LABEL_CLUSTER_VALUES is not None:
        label_gdf = city_gdf[
            city_gdf["cluster"].isin(LABEL_CLUSTER_VALUES)
        ].copy()
    else:
        label_gdf = city_gdf.copy()

    label_gdf = label_gdf[
        label_gdf.geometry.notnull() &
        (~label_gdf.geometry.is_empty)
    ].copy()

    label_gdf["_label_priority"] = np.where(label_gdf["cluster"] == 0, 1, 0)
    label_gdf = label_gdf.sort_values(
        by=["_label_priority", "cluster"],
        ascending=[True, True]
    )

    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    ax_bbox = ax.get_window_extent(renderer)

    placed_bboxes = []

    placed_bboxes.extend(
        make_point_obstacles(
            ax=ax,
            gdf=base_point_gdf,
            radius_px=BASE_POINT_OBSTACLE_RADIUS_PX
        )
    )

    placed_bboxes.extend(
        make_point_obstacles(
            ax=ax,
            gdf=overlay_point_gdf,
            radius_px=OVERLAY_POINT_OBSTACLE_RADIUS_PX
        )
    )

    placed_count = 0
    skipped_count = 0

    for _, row in label_gdf.iterrows():
        name = row[name_field]
        label_text = format_city_label(name)

        if label_text == "":
            continue

        x0 = row.geometry.x
        y0 = row.geometry.y

        placed = False

        for dx, dy, ha, va in LABEL_CANDIDATE_OFFSETS:

            if CITY_LABEL_BBOX:
                bbox_style = dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=CITY_LABEL_BBOX_ALPHA,
                    pad=0.15
                )
            else:
                bbox_style = None

            txt = ax.text(
                x0 + dx,
                y0 + dy,
                label_text,
                fontsize=CITY_LABEL_SIZE,
                color=CITY_LABEL_COLOR,
                ha=ha,
                va=va,
                bbox=bbox_style,
                zorder=18
            )

            fig.canvas.draw()
            bbox = txt.get_window_extent(renderer).expanded(
                LABEL_BBOX_EXPAND_X,
                LABEL_BBOX_EXPAND_Y
            )

            if KEEP_LABELS_INSIDE_AXES and not bbox_inside_axes(bbox, ax_bbox):
                txt.remove()
                continue

            has_overlap = any(
                bbox.overlaps(old_bbox)
                for old_bbox in placed_bboxes
            )

            if has_overlap:
                txt.remove()
                continue

            placed_bboxes.append(bbox)
            placed = True
            placed_count += 1
            break

        if not placed:
            skipped_count += 1

            if not SKIP_OVERLAPPED_LABELS:
                dx, dy, ha, va = LABEL_CANDIDATE_OFFSETS[0]
                txt = ax.text(
                    x0 + dx,
                    y0 + dy,
                    label_text,
                    fontsize=CITY_LABEL_SIZE,
                    color=CITY_LABEL_COLOR,
                    ha=ha,
                    va=va,
                    zorder=18
                )
                fig.canvas.draw()
                bbox = txt.get_window_extent(renderer).expanded(
                    LABEL_BBOX_EXPAND_X,
                    LABEL_BBOX_EXPAND_Y
                )
                placed_bboxes.append(bbox)

    print(f"城市名标注完成：成功 {placed_count} 个，跳过 {skipped_count} 个。")


def add_cluster_legend(ax):
    """
    设置 cluster 与 outlier 图例：
    C1–C4 使用圆形符号；
    Outlier 使用空心菱形；
    No composite heatwave 使用灰色方块。
    """
    handles = []

    for cls in [1, 2, 3, 4]:
        style = BASE_POINT_STYLES[cls]

        handle = Line2D(
            [0],
            [0],
            marker=style["marker"],
            linestyle="None",
            label=style["label"],
            markerfacecolor=style["facecolor"],
            markeredgecolor=style["edgecolor"],
            markeredgewidth=style["linewidth"],
            markersize=LEGEND_MARKER_SIZE
        )
        handles.append(handle)

    outlier_handle = Line2D(
        [0],
        [0],
        marker="D",
        linestyle="None",
        label="Outlier reassigned by shape",
        markerfacecolor="#FFFFFF",
        markeredgecolor="#333333",
        markeredgewidth=1.1,
        markersize=LEGEND_MARKER_SIZE
    )
    handles.append(outlier_handle)

    no_heatwave_style = BASE_POINT_STYLES[0]
    no_heatwave_handle = Line2D(
        [0],
        [0],
        marker=no_heatwave_style["marker"],
        linestyle="None",
        label=no_heatwave_style["label"],
        markerfacecolor=no_heatwave_style["facecolor"],
        markeredgecolor=no_heatwave_style["edgecolor"],
        markeredgewidth=no_heatwave_style["linewidth"],
        markersize=LEGEND_MARKER_SIZE
    )
    handles.append(no_heatwave_handle)

    leg = ax.legend(
        handles=handles,
        loc=LEGEND_LOC,
        bbox_to_anchor=LEGEND_BBOX,
        fontsize=LEGEND_FONT_SIZE,
        frameon=LEGEND_FRAME,
        framealpha=LEGEND_FRAME_ALPHA,
        edgecolor=LEGEND_EDGE_COLOR,
        borderpad=0.6,
        handletextpad=0.7,
        labelspacing=0.45
    )

    leg.set_zorder(40)


# =========================================================
# 5. 读取数据
# =========================================================

outline_gdf = gpd.read_file(gdb_path, layer=outline_layer)
states_gdf = gpd.read_file(states_shp)

outline_wgs84 = outline_gdf.to_crs(epsg=4326)
states_wgs84 = states_gdf.to_crs(epsg=4326)

outline_5070 = outline_wgs84.to_crs(TARGET_CRS)
states_5070 = states_wgs84.to_crs(TARGET_CRS)

xlim, ylim = get_display_xlim_ylim(outline_5070)

city_gdf = gpd.read_file(city_cluster_path)
city_gdf = city_gdf.to_crs(TARGET_CRS)

overlay_gdf = gpd.read_file(overlay_gdb_path, layer=overlay_layer)
overlay_gdf = overlay_gdf.to_crs(TARGET_CRS)

if "cluster" not in city_gdf.columns:
    raise ValueError("圆形城市点数据中未找到字段：cluster")

if "cluster" not in overlay_gdf.columns:
    raise ValueError("棱形覆盖点数据中未找到字段：cluster")

if CITY_NAME_FIELD not in city_gdf.columns:
    raise ValueError(f"圆形城市点数据中未找到城市名字段：{CITY_NAME_FIELD}")

city_gdf["cluster"] = pd.to_numeric(
    city_gdf["cluster"],
    errors="coerce"
).astype("Int64")

overlay_gdf["cluster"] = pd.to_numeric(
    overlay_gdf["cluster"],
    errors="coerce"
).astype("Int64")

city_gdf = city_gdf.dropna(subset=["cluster"]).copy()
overlay_gdf = overlay_gdf.dropna(subset=["cluster"]).copy()

city_gdf["cluster"] = city_gdf["cluster"].astype(int)
overlay_gdf["cluster"] = overlay_gdf["cluster"].astype(int)


# =========================================================
# 6. 绘图
# =========================================================

fig, ax = plt.subplots(
    figsize=(FIG_WIDTH, FIG_HEIGHT),
    dpi=DPI,
    facecolor="white"
)

ax.set_facecolor("white")
ax.set_xlim(xlim)
ax.set_ylim(ylim)

add_graticules(ax, TARGET_CRS)

# ---------- 美国底图：浅灰色填充 ----------
states_5070.plot(
    ax=ax,
    facecolor=USA_FACE_COLOR,
    edgecolor=USA_EDGE_COLOR,
    linewidth=STATE_LINE_WIDTH,
    zorder=2
)

# ---------- 美国外边界 ----------
outline_5070.boundary.plot(
    ax=ax,
    color=OUTLINE_COLOR,
    linewidth=OUTLINE_WIDTH,
    zorder=4
)

# ---------- 圆形城市点 ----------
for cls, style in BASE_POINT_STYLES.items():
    sub = city_gdf[city_gdf["cluster"] == cls]

    if len(sub) == 0:
        continue

    ax.scatter(
        sub.geometry.x,
        sub.geometry.y,
        s=style["size"],
        marker=style["marker"],
        facecolor=style["facecolor"],
        edgecolor=style["edgecolor"],
        linewidth=style["linewidth"],
        alpha=1.0,
        zorder=10
    )

# ---------- 城市名称，避让圆点和菱形 ----------
if SHOW_CITY_LABELS:
    add_city_labels_greedy(
        ax=ax,
        city_gdf=city_gdf,
        name_field=CITY_NAME_FIELD,
        base_point_gdf=city_gdf,
        overlay_point_gdf=overlay_gdf
    )

# ---------- 棱形覆盖点，最上层 ----------
for cls, style in OVERLAY_POINT_STYLES.items():
    sub = overlay_gdf[overlay_gdf["cluster"] == cls]

    if len(sub) == 0:
        continue

    ax.scatter(
        sub.geometry.x,
        sub.geometry.y,
        s=style["size"],
        marker=style["marker"],
        facecolor=style["facecolor"],
        edgecolor=style["edgecolor"],
        linewidth=style["linewidth"],
        alpha=1.0,
        zorder=25
    )

if SHOW_SCALEBAR:
    add_scale_bar(ax)

if SHOW_LEGEND:
    add_cluster_legend(ax)

for spine in ax.spines.values():
    spine.set_linewidth(SPINE_WIDTH)
    spine.set_color("black")


# =========================================================
# 7. 保存
# =========================================================

plt.subplots_adjust(
    left=0.075,
    right=0.995,
    bottom=0.08,
    top=0.99
)

if SAVE_PNG:
    out_png = out_dir / f"{output_name}.png"
    plt.savefig(
        out_png,
        dpi=DPI,
        facecolor="white"
    )
    print(f"PNG 已保存：{out_png}")

if SAVE_SVG:
    out_svg = out_dir / f"{output_name}.svg"
    plt.savefig(
        out_svg,
        format="svg",
        facecolor="white"
    )
    print(f"SVG 已保存：{out_svg}")

plt.show()
plt.close()